# Speech Data — прогресс и проверка данных
Единый рабочий ноутбук. **Run → Run All Cells** читает свежие данные из S3, без запросов к Gemini.

1. Прогресс, часы аудио и выход отбора по корпусам.
2. Каталоги, новые поступления, аккаунты и ошибки.
3. Фильтры примеров, исходные метки, три ответа и аудио.
4. Фиксированные примеры с замечаниями после прослушивания.
5. Исходные объекты, сырые запросы и сохранённые эксперименты с промптами.

Данные обновляются Airflow примерно раз в пять минут. Кандидаты 3/3 не равны human-gold. Исторические эксперименты включаются отдельно в последнем разделе.


In [ ]:
import json, hashlib
import pandas as pd
from IPython.display import Audio, display
from lalm.annotation.inspection import open_snapshot
from lalm.annotation.contract import TRIAD_VERSION, triad_decision
if 'db' in globals():
    db.close()
    cache.cleanup()
s3, snapshot, cache, db = open_snapshot()
print('Снимок:', snapshot['snapshot_at'])
age = pd.Timestamp.now(tz='UTC') - pd.Timestamp(snapshot['snapshot_at'])
print('Возраст снимка:', round(age.total_seconds()/60, 1), 'мин')
if age.total_seconds() > 900:
    print('ВНИМАНИЕ: снимок старше 15 минут — проверьте Airflow.')
passes = db.sql('SELECT * FROM passes').df()
mask = passes.batch_id.fillna('').str.startswith(TRIAD_VERSION+'-')
current = passes[mask].copy()
examples = db.sql('SELECT * FROM examples').df()
examples = examples[examples.batch_id.fillna('').str.startswith(TRIAD_VERSION+'-')].copy()
old_texts = passes[~mask & passes.gemini_text.notna()].groupby('sample_id').gemini_text.agg(list).to_dict()
records = []
for (batch, sample), group in current.groupby(['batch_id','sample_id']):
    results = {}
    for r in group.itertuples():
        if isinstance(r.gemini_text, str):
            spans = json.loads(r.uncertain_spans) if isinstance(r.uncertain_spans, str) else []
            results[r.prompt_variant] = {'verbatim_text': r.gemini_text, 'language': r.language or '', 'uncertain_spans': spans}
    decision = triad_decision(results, old_texts.get(sample, []))
    if decision == 'waiting_remaining' and group.original_annotation_status.eq('not_linked_or_not_available').all():
        decision = 'missing_source_annotation'
    if (group.processing_state.isin(['NEEDS_REVIEW','PERMANENT_FAILED']) & group.gemini_text.isna()).any():
        decision = 'rejected_failed_pass'
    records.append({'batch_id':batch, 'sample_id':sample, 'decision':decision})
status = pd.DataFrame(records, columns=['batch_id','sample_id','decision'])
examples = examples.merge(status, on=['batch_id','sample_id'], how='left', validate='one_to_one')
labels = {'candidate_3of3':'Совпали 3/3: кандидаты', 'waiting_first':'Ожидают первый ответ',
          'waiting_remaining':'Ожидают остальные проходы', 'missing_source_annotation':'Нет исходной метки',
          'rejected_first':'Первый ответ: не ru / неопределённость / пусто',
          'rejected_quality':'Не прошли проверку языка / определённости',
          'rejected_disagreement':'Расхождение трёх ответов', 'rejected_prior_conflict':'Конфликт с прежними ответами',
          'rejected_failed_pass':'Ошибка или отклонённый ответ'}
examples['статус'] = examples.decision.map(labels).fillna(examples.decision)


## Сколько обработано
Сводка относится к текущему протоколу precision-ru-triad-v1. Полного числа будущих фрагментов ещё нет: архивы продолжают разбираться, поэтому общий процент готовности не вычисляется.

In [ ]:
summary = {
    'Уникальных аудио в очереди': current.audio_sha256.nunique(),
    'Всего заданий-проходов': len(current),
    'Сохранённых текстовых ответов': int(current.gemini_text.notna().sum()),
    'В работе (на момент снимка)': int(current.processing_state.eq('RUNNING').sum()),
    'Ожидают / повтор после ошибки': int(current.processing_state.isin(['PENDING','RETRYABLE_FAILED']).sum()),
    'Уникальных аудио — кандидаты 3/3': examples.loc[examples.decision.eq('candidate_3of3'),'audio_sha256'].nunique(),
    'Подтверждено человеком': int(examples.human_verified.sum()),
}
display(pd.Series(summary, name='Количество').to_frame())
display(examples.groupby('статус', dropna=False).size().rename('Примеров').to_frame())
display(pd.crosstab(examples.corpus, examples['статус']))


## Часы аудио
Длительность берётся из сохранённых ответов и считается **один раз на SHA256 аудио**, независимо от числа проходов. У ожидающих аудио без ответа длительность в снимке пока неизвестна: они показаны отдельно, а не приняты за нулевую длительность.

Это часы подготовленных фрагментов: перекрытия соседних сегментов не вычитаются. «Кандидаты 3/3» ещё не означают проверенную человеком разметку.


In [ ]:
durations = current.groupby('audio_sha256', dropna=True).audio_seconds.max()
audio_rows = examples.sort_values('updated_at').drop_duplicates('audio_sha256', keep='last').copy()
audio_rows['audio_seconds'] = audio_rows.audio_sha256.map(durations)
with_text = set(current.loc[current.gemini_text.notna(), 'audio_sha256'])
audio_rows['has_answer'] = audio_rows.audio_sha256.isin(with_text)
audio_rows['candidate'] = audio_rows.decision.eq('candidate_3of3')
hours = pd.Series({
    'Известная длительность уникальных фрагментов, ч': audio_rows.audio_seconds.sum()/3600,
    'Есть хотя бы один сохранённый ответ, ч': audio_rows.loc[audio_rows.has_answer, 'audio_seconds'].sum()/3600,
    'Кандидаты 3/3, ч': audio_rows.loc[audio_rows.candidate, 'audio_seconds'].sum()/3600,
    'Подтверждено человеком, ч': audio_rows.loc[audio_rows.human_verified, 'audio_seconds'].sum()/3600,
}, name='Часы')
display(hours.round(3).to_frame())
print('Аудио с неизвестной длительностью:', int(audio_rows.audio_seconds.isna().sum()))
by_corpus = audio_rows.assign(
    known_hours=audio_rows.audio_seconds/3600,
    answered_hours=audio_rows.audio_seconds.where(audio_rows.has_answer, 0)/3600,
    candidate_hours=audio_rows.audio_seconds.where(audio_rows.candidate, 0)/3600,
    unknown_duration=audio_rows.audio_seconds.isna(),
).groupby('corpus', dropna=False).agg(
    unique_audio=('audio_sha256','size'),
    known_hours=('known_hours','sum'),
    answered_hours=('answered_hours','sum'),
    candidate_3of3_hours=('candidate_hours','sum'),
    unknown_duration=('unknown_duration','sum'),
)
display(by_corpus.round(3))
display(audio_rows.groupby('статус', dropna=False).agg(
    unique_audio=('audio_sha256','size'), seconds=('audio_seconds','sum')
).assign(hours=lambda x:x.seconds/3600).drop(columns='seconds').round(3))


## Выход отбора по корпусам
Доля кандидатов 3/3 среди завершённых проверок. Ошибки или отклонённые ответы не считаются браком аудио.


In [ ]:
# Only explicit quality decisions enter the acceptance denominator.
quality_decisions = {'candidate_3of3', 'rejected_first', 'rejected_quality',
                     'rejected_disagreement', 'rejected_prior_conflict'}
yield_rows = audio_rows.copy()
yield_rows['completed'] = yield_rows.decision.isin(quality_decisions)
yield_rows['waiting'] = yield_rows.decision.isin(['waiting_first', 'waiting_remaining'])
yield_rows['failed'] = yield_rows.decision.eq('rejected_failed_pass')
yield_rows['missing_source'] = yield_rows.decision.eq('missing_source_annotation')
yield_rows['completed_hours'] = yield_rows.audio_seconds.where(yield_rows.completed, 0)/3600
yield_rows['accepted_hours'] = yield_rows.audio_seconds.where(yield_rows.candidate, 0)/3600
yield_rows['completed_unknown_duration'] = yield_rows.completed & yield_rows.audio_seconds.isna()
yield_by_corpus = yield_rows.groupby('corpus', dropna=False).agg(
    registered=('audio_sha256', 'size'), completed=('completed', 'sum'),
    accepted=('candidate', 'sum'), waiting=('waiting', 'sum'),
    failed_or_rejected_response=('failed', 'sum'), missing_source=('missing_source', 'sum'),
    completed_hours=('completed_hours', 'sum'), accepted_hours=('accepted_hours', 'sum'),
    completed_unknown_duration=('completed_unknown_duration', 'sum'))
yield_by_corpus['accepted_pct_samples'] = 100*yield_by_corpus.accepted/yield_by_corpus.completed.where(yield_by_corpus.completed.ne(0))
yield_by_corpus['accepted_pct_known_hours'] = 100*yield_by_corpus.accepted_hours/yield_by_corpus.completed_hours.where(yield_by_corpus.completed_hours.ne(0))
display(yield_by_corpus.round(2))
print('Знаменатель: завершённые решения по качеству; ожидание, ошибки ответа и отсутствие метки исключены.')
print('registered — только заведённые в этот прогон фрагменты, не полный объём исходного корпуса.')
print('Проценты по часам используют известную длительность; нулевой знаменатель даёт NaN.')
print('Совпадение 3/3 — кандидат, а не подтверждённая человеком точность.')


## Датасеты: общий объём и новые поступления
Каталог ниже читается **напрямую из S3 при Run All** и включает новые загрузки, даже если они ещё не попали в очередь Airflow. Данные очереди относятся к времени снимка выше.

Часы исходников — отдельные оценки из метаданных: источник и границы оценки указаны рядом. `NaN` означает «не измерено», а не 0. Строки могут описывать перекрывающиеся копии или части датасета — **их часы нельзя складывать**. Общие часы исходников и часы нарезанных фрагментов выше имеют разный смысл.

Airflow каждые 5 минут обнаруживает поддерживаемые файлы в `speech-corpora/staging/`. Частные архивы и benchmark не включаются автоматически. `tar.zst` и вложенные ZIP требуют отдельной подготовки.


In [ ]:
from lalm.annotation.inspection import read_text_object
catalog_bucket = 'speech-datasets'
objects = [obj for page in s3.get_paginator('list_objects_v2').paginate(Bucket=catalog_bucket, Prefix='catalog/') for obj in page.get('Contents', [])]
index = {o['Key']:o for o in objects}
def read_catalog(key):
    return json.loads(read_text_object(s3, catalog_bucket, key))
receipts = []
for obj in objects:
    if obj['Key'].endswith('/status.json'):
        d = read_catalog(obj['Key'])
        receipts.append({'dataset':obj['Key'].split('/')[-2], 'storage_status':d.get('status'),
                         'receipt_updated':obj['LastModified'], 'source_use':d.get('source_use'),
                         'archive_format':d.get('archive_format'), 'receipt':obj['Key']})
receipts = pd.DataFrame(receipts)
print('Последние поступления / обновления каталога (время записи отчёта в S3):')
display(receipts.sort_values('receipt_updated', ascending=False).reset_index(drop=True))

catalog_rows = []
keys = ['catalog/native-corpora-20261012/native-corpus-catalog.json',
        'catalog/full-download-20261012/full-download-plan.json',
        'catalog/full-download-20261012/russian-compressed-sources-20261013.json']
for key in keys:
    if key not in index:
        continue
    doc = read_catalog(key)
    entries = doc.get('corpora', doc.get('sources', []))
    if not isinstance(entries, list):
        continue
    for entry in entries:
        if not isinstance(entry, dict):
            continue
        name = entry.get('id', entry.get('name', entry.get('source', 'unknown')))
        uri = next((entry[k] for k in ('s3','s3_target','native_audio_prefix','native_prefix','audio')
                    if isinstance(entry.get(k),str) and entry[k].startswith('s3://')), None)
        corpus = uri.split('/')[4] if uri and uri.startswith('s3://speech-corpora/staging/') else None
        h = None; evidence = 'Длительность не указана'; scope = entry.get('source_selection', entry.get('selection', 'см. каталог'))
        if isinstance(entry.get('producer_reported_hours'), (int,float)):
            h = entry['producer_reported_hours']; evidence = 'Заявлено производителем копии'
        elif isinstance(entry.get('audio_hours'), (int,float)):
            h = entry['audio_hours']; evidence = 'audio_hours из каталога'
        elif isinstance(entry.get('reported_minutes'), (int,float)):
            h = entry['reported_minutes']/60; evidence = 'Минуты издателя / 60'
        elif isinstance(entry.get('publisher_rounded_pilot_hours'), (int,float)):
            h = entry['publisher_rounded_pilot_hours']; evidence = 'Округлённая оценка издателя'; scope = 'Только скачанный pilot, не весь датасет'
        elif isinstance(entry.get('inspected_train_yaml'),dict):
            h = entry['inspected_train_yaml'].get('hours'); evidence = 'Сумма длительностей train.yaml'; scope = 'Только train-сегменты, не весь архив'
        receipt_uri = entry.get('receipt','')
        receipt_key = receipt_uri.removeprefix('s3://speech-datasets/')
        receipt = read_catalog(receipt_key) if receipt_key in index else {}
        state = receipt.get('status', entry.get('status', entry.get('status_at_launch','unknown')))
        catalog_rows.append({'dataset':name, 'corpus_in_queue':corpus, 'source_hours':h,
                             'hours_evidence':evidence, 'scope':scope, 'storage_status':state,
                             'source_uri':uri, 'catalog':key})
catalog_table = pd.DataFrame(catalog_rows)
sources = db.sql('SELECT * FROM sources').df()
queue_inventory = sources.groupby('corpus', dropna=False).agg(
    registered_objects=('source_id','size'), registered_bytes=('bytes','sum'),
    first_seen=('created_at','min'), last_arrival=('created_at','max'),
    last_activity=('updated_at','max'))
queue_inventory['registered_GiB'] = queue_inventory.registered_bytes/1024**3
catalog_table = catalog_table.merge(queue_inventory[['registered_objects','last_arrival']],
                                    left_on='corpus_in_queue', right_index=True, how='left')
print('Известные часы исходников и охват очередью:')
with pd.option_context('display.max_colwidth', 100):
    display(catalog_table)
print('Все корпуса в очереди, включая отсутствующие в каталогах:')
display(queue_inventory.drop(columns='registered_bytes').join(by_corpus, how='left').sort_values('last_arrival', ascending=False))
display(pd.crosstab(sources.corpus, sources.preparation_state))

# Прямое сравнение названий префиксов S3 с текущим снимком очереди.
live_prefixes = [x['Prefix'] for page in s3.get_paginator('list_objects_v2').paginate(
    Bucket='speech-corpora', Prefix='staging/', Delimiter='/') for x in page.get('CommonPrefixes', [])]
live_corpora = {p.split('/')[1] for p in live_prefixes}
missing = sorted(live_corpora - set(sources.corpus.dropna()))
print('Префиксы S3, ещё отсутствующие в снимке очереди:', missing or 'нет')
print('Совпадение названий не доказывает регистрацию каждого объекта. Новые tar.zst видны в отчётах загрузки, но не поддерживаются текущим preparer.')


## Аккаунты, ошибки и недавние ответы
Это активность из снимка, а не остаток квоты. NEEDS_REVIEW с pilot_review_only — нормальный сохранённый результат, ожидающий проверки. Число ответов за интервал считается по updated_at, а не по полной истории событий.

In [ ]:
display(current.groupby(['account_id','processing_state'], dropna=False).agg(
    requests=('job_id','size'), texts=('gemini_text','count'), last_update=('updated_at','max')))
errors = current[current.error_code.notna() & current.error_code.ne('pilot_review_only')]
display(errors.groupby(['account_id','error_code'], dropna=False).size().rename('Запросов').to_frame())
answered = current[current.gemini_text.notna()].copy()
answered['интервал UTC'] = pd.to_datetime(answered.updated_at, utc=True).dt.floor('30min')
display(answered.groupby('интервал UTC').size().rename('Ответов').tail(24).to_frame())


## Операционный монитор: очередь, скорость, ошибки и аккаунты
Airflow публикует компактный снимок каждые 5 минут в `speech-datasets/analytics/ops/`. В нём нет аудио и транскриптов. Скорость считается по сохранённым выходам; `pilot_review_only` и отклонения `non_russian` показаны отдельно от операционных ошибок.


In [ ]:
import pandas as pd
from lalm.annotation.inspection import read_text_object

ops_bucket = 'speech-datasets'
ops_key = 'analytics/ops/latest.json'
ops = json.loads(read_text_object(s3, ops_bucket, ops_key))
ops_at = pd.Timestamp(ops['captured_at'])
print('Снимок:', ops_at.tz_convert('Europe/Moscow').strftime('%Y-%m-%d %H:%M:%S МСК'),
      '| возраст:', round((pd.Timestamp.now(tz='UTC') - ops_at).total_seconds() / 60, 1), 'мин')

global_ops = pd.Series(ops['global'])
display(global_ops.to_frame('Значение'))
accounts_ops = pd.DataFrame(ops['accounts'])
account_columns = ['id','enabled','blocked_reason','concurrency','running','saved_5m',
                   'saved_per_minute_15m','median_success_seconds_15m','errors_5m',
                   'errors_15m','ambiguous_timeouts_60m','review_outputs_15m','rejected_language_15m','cooldown_seconds']
display(accounts_ops[account_columns].sort_values(['enabled','id'], ascending=[False,True])
        .rename(columns={'id':'Аккаунт','enabled':'Включён','blocked_reason':'Причина паузы',
                         'concurrency':'Параллельность','running':'Выполняется','saved_5m':'Сохранено за 5 мин',
                         'saved_per_minute_15m':'Выходов/мин за 15 мин','median_success_seconds_15m':'Медиана, сек',
                         'errors_5m':'Ошибок за 5 мин','errors_15m':'Ошибок за 15 мин',
                         'ambiguous_timeouts_60m':'Неоднозначных таймаутов за час',
                         'review_outputs_15m':'Ожидают проверки за 15 мин','rejected_language_15m':'Отказано: не русский за 15 мин',
                         'cooldown_seconds':'Cooldown, сек'}))
recent_ops_errors = pd.DataFrame(ops['recent_errors'])
print('Последние служебные ошибки (без текстов ответов):')
display(recent_ops_errors)

# Trend chart: fetch only the last six hours of compact operational snapshots.
ops_start = pd.Timestamp.now(tz='UTC') - pd.Timedelta(hours=6)
ops_objects = []
for day in pd.date_range(ops_start.normalize(), pd.Timestamp.now(tz='UTC').normalize(), freq='D'):
    prefix = f"analytics/ops/history/{day:%Y/%m/%d}/"
    for page in s3.get_paginator('list_objects_v2').paginate(Bucket=ops_bucket, Prefix=prefix):
        ops_objects.extend(o for o in page.get('Contents', []) if o['LastModified'] >= ops_start.to_pydatetime())
ops_objects = sorted(ops_objects, key=lambda o:o['LastModified'])[-72:]
ops_history = [json.loads(read_text_object(s3, ops_bucket, o['Key'])) for o in ops_objects]
if ops_history:
    trend = pd.DataFrame([{'at': pd.Timestamp(x['captured_at']),
                           'saved_per_minute': x['global']['saved_5m'] / 5,
                           'running': x['global']['running'], 'queued': x['global']['queued'],
                           'expired_leases': x['global']['expired_leases']} for x in ops_history]).set_index('at')
    display(trend.tail(20))
    display(trend[['saved_per_minute','running']].plot(figsize=(12,4), title='Скорость сохранения и активные задания, последние 6 часов'))
else:
    print('Исторические снимки пока не накопились; график появится после нескольких запусков Airflow.')


## Входные данные и публикации
Исходный объект может быть архивом с множеством аудио. Его нельзя считать одним готовым примером. Пустые публикации означают, что обучающий релиз ещё не выпущен.

In [ ]:
display(db.sql("SELECT corpus, preparation_state, count(*) AS source_objects, sum(samples_prepared) AS prepared_samples, sum(preparation_errors) AS preparation_errors FROM sources GROUP BY ALL ORDER BY corpus, preparation_state").df())
display(db.sql('SELECT * FROM publications ORDER BY created_at DESC').df())


## Выбрать примеры
По умолчанию показаны последние кандидаты, где совпали все три ответа. Измените фильтры и запустите эту ячейку и следующую. Для всех статусов поставьте decision_filter = None; для расхождений — rejected_disagreement.

In [ ]:
decision_filter = 'candidate_3of3'
corpus_filter = None
sample_search = ''  # Полный ID или его начало.
limit = 30
selected = examples.copy()
if decision_filter:
    selected = selected[selected.decision.eq(decision_filter)]
if corpus_filter:
    selected = selected[selected.corpus.eq(corpus_filter)]
if sample_search:
    selected = selected[selected.sample_id.str.startswith(sample_search, na=False)]
print('Найдено:', len(selected))
selected = selected.sort_values('updated_at', ascending=False).head(limit).reset_index(drop=True)
columns = ['sample_id','corpus','статус','original_annotation_json','gemini_verbatim',
           'gemini_coverage','gemini_source_correction','passes_with_text','unfinished_passes','error_codes']
with pd.option_context('display.max_colwidth', 180):
    display(selected[columns])


## Прослушать и сравнить
Выберите номер строки из таблицы выше. Плеер загружает только выбранный фрагмент и проверяет SHA256. Все три текста показываются полностью.

In [ ]:
example_index = 0
if selected.empty:
    print('Для этих фильтров примеров пока нет.')
else:
    row = selected.iloc[example_index]
    print('Пример:', row.sample_id, '|', row['статус'])
    print('Исходная метка:', row.original_annotation_json)
    group = current[(current.batch_id == row.batch_id) & (current.sample_id == row.sample_id)]
    for item in group.sort_values('prompt_variant').itertuples():
        print('\n', item.prompt_variant, '| язык:', item.language, '|', item.processing_state, '|', item.error_code)
        print(item.gemini_text if isinstance(item.gemini_text, str) else 'Нет принятого текста')
    response = s3.get_object(Bucket=row.audio_bucket, Key=row.audio_key)
    try:
        if response['ContentLength'] > 100 * 1024**2:
            raise ValueError('Фрагмент слишком большой для плеера')
        audio = response['Body'].read()
    finally:
        response['Body'].close()
    assert hashlib.sha256(audio).hexdigest() == row.audio_sha256
    display(Audio(audio))
    print('Источник и границы фрагмента:')
    display(json.loads(group.iloc[0].source_provenance or '{}'))


## Прямой API: отдельные партии для проверки на слух

По умолчанию показана последняя завершённая регулярная API shadow-партия. Можно выбрать `four_account` (30 аудио × 4 аккаунта) или `shadow_01` (100 аудио на одном аккаунте). `candidate_3of3` означает согласие трёх ответов **внутри аккаунта**, а не проверенную на слух метку. В обоих режимах результаты остаются вне обучающей публикации.


In [ ]:
native_run = 'latest_recurring'  # 'latest_recurring', 'four_account' или 'shadow_01'
native_runs = {
    'four_account': ('evaluation/native-api-canary-v1/20261014-expanded-01',
                     ['agy-03', 'agy-16', 'agy-08', 'agy-21']),
    'shadow_01': ('evaluation/native-api-canary-v1/20261014-shadow-01', ['agy-03']),
}
if native_run == 'latest_recurring':
    root = 'evaluation/native-api-canary-v1/recurring/'
    pages = s3.get_paginator('list_objects_v2').paginate(Bucket='speech-annotations', Prefix=root)
    completed = sorted(obj['Key'] for page in pages for obj in page.get('Contents', [])
                       if obj['Key'].endswith('/agy-03/results.json'))
    if not completed:
        raise RuntimeError('Пока нет завершённых API shadow-партий')
    native_prefix = completed[-1].removesuffix('/agy-03/results.json')
    native_accounts = ['agy-03']
else:
    native_prefix, native_accounts = native_runs[native_run]
print('API shadow:', native_prefix)
native_reports = {}
for account in native_accounts:
    obj = s3.get_object(Bucket='speech-annotations', Key=f'{native_prefix}/{account}/results.json')
    try:
        native_reports[account] = json.loads(obj['Body'].read())
    finally:
        obj['Body'].close()
rows = []
for i, base in enumerate(native_reports[native_accounts[0]]['comparisons']):
    sha = base['audio_sha256']
    decisions = [native_reports[a]['comparisons'][i]['native_decision'] for a in native_accounts]
    assert all(native_reports[a]['comparisons'][i]['audio_sha256'] == sha for a in native_accounts)
    row = {'audio_sha256':sha, 'audit':base.get('control', {}).get('audit', ''),
           'все_выбранные_приняли':all(x == 'candidate_3of3' for x in decisions)}
    for account, decision in zip(native_accounts, decisions):
        first = native_reports[account]['comparisons'][i]['native_passes'].get('verbatim', {})
        row[account+'_решение'] = decision
        row[account+'_текст'] = first.get('verbatim_text')
    rows.append(row)
native_audit = pd.DataFrame(rows).sort_values('все_выбранные_приняли', ascending=False).reset_index(drop=True)
print('Всего аудио:', len(native_audit), '| кандидаты у выбранных аккаунтов:', int(native_audit['все_выбранные_приняли'].sum()))
with pd.option_context('display.max_colwidth', 140):
    display(native_audit[['audio_sha256','audit','все_выбранные_приняли'] +
                         [a+'_решение' for a in native_accounts] +
                         [a+'_текст' for a in native_accounts]])


### Прослушать выбранное аудио из эксперимента

Плеер читает исходный объект только для выбранной строки и сверяет SHA256. Прослушивание само по себе не ставит `human_verified` и не публикует метку в обучение.


In [ ]:
native_example_index = 0
if not 0 <= native_example_index < len(native_audit):
    raise IndexError('Выберите существующий номер строки в native_audit')
choice = native_audit.iloc[native_example_index]
match = examples[examples.audio_sha256.eq(choice.audio_sha256)]
if match.empty:
    print('Аудио ещё нет в текущем снимке данных. Обновите ноутбук после следующего Airflow snapshot.')
else:
    source = match.sort_values('updated_at').iloc[-1]
    print('SHA256:', choice.audio_sha256, '| корпус:', source.corpus)
    print('Исходная метка:', source.original_annotation_json)
    for account in native_accounts:
        print(account, '|', choice[account+'_решение'], '|', choice[account+'_текст'])
    response = s3.get_object(Bucket=source.audio_bucket, Key=source.audio_key)
    try:
        if response['ContentLength'] > 100 * 1024**2:
            raise ValueError('Фрагмент слишком большой для плеера')
        audio = response['Body'].read()
    finally:
        response['Body'].close()
    assert hashlib.sha256(audio).hexdigest() == choice.audio_sha256
    display(Audio(audio))


## Примеры для проверки ушами — 13 октября
Восемь совпадений 3/3 из свежих ответов. Это диагностическая, не случайная репрезентативная выборка. Первым — «переоформление внука» → «У меня фамилия Азбука». Ни исходная метка, ни согласие модели не считаются эталоном.
Выполните **Run → Run All**; ниже у каждого примера есть плеер и три ответа.

**Подтверждение пользователя после прослушивания, 13 октября:** для аудио `3bda4d2da118…` верно «У меня фамилия Азбука»; исходная метка «переоформление внука» неверна. Это отзыв по одному примеру, не оценка точности корпуса.


In [ ]:
# Fixed examples from the text sanity check on 2026-10-13.
audit_prefixes = ['3bda4d2da118', 'ea16491d1084', '1ee4be134620', '5173f3f5724a',
                  'bef15e4dc234', 'ca3b33b8fc05', 'a6e518d476a7', 'f6842eda42d5']
for prefix in audit_prefixes:
    matches = examples[examples.audio_sha256.str.startswith(prefix, na=False)]
    if matches.empty:
        print(prefix, '— ещё нет в снимке; повторите Run All после обновления Airflow.')
        continue
    if matches.audio_sha256.nunique() != 1:
        raise ValueError('Неоднозначный префикс SHA256: '+prefix)
    row = matches.sort_values('updated_at').iloc[-1]
    print('\n'+'='*70)
    print('Аудио:', row.audio_sha256, '|', row.corpus, '|', row['статус'])
    print('Исходная метка:', row.original_annotation_json)
    group = current[(current.batch_id == row.batch_id) & (current.sample_id == row.sample_id)]
    for variant in ('verbatim', 'coverage', 'source_correction'):
        for item in group[group.prompt_variant.eq(variant)].itertuples():
            print(variant, '| язык:', item.language, '|', item.processing_state)
            print(item.gemini_text if isinstance(item.gemini_text, str) else 'Нет принятого текста')
    response = s3.get_object(Bucket=row.audio_bucket, Key=row.audio_key)
    try:
        if response['ContentLength'] > 100 * 1024**2:
            raise ValueError('Фрагмент слишком большой для плеера')
        audio = response['Body'].read()
    finally:
        response['Body'].close()
    assert hashlib.sha256(audio).hexdigest() == row.audio_sha256
    display(Audio(audio))
print('Прослушивание здесь само по себе не меняет human_verified и не публикует данные для обучения.')


## Исходные объекты и сырые ответы выбранного примера


In [ ]:
from pathlib import Path
from lalm.annotation.inspection import read_text_object

def inspect_source(source_bucket, source_key, play=False):
    meta = s3.head_object(Bucket=source_bucket, Key=source_key)
    print({'uri': f's3://{source_bucket}/{source_key}', 'bytes': meta['ContentLength']})
    if play:
        if Path(source_key).suffix.lower() not in {'.wav','.mp3','.opus','.ogg','.flac','.m4a'}:
            raise ValueError('Выберите аудиофрагмент, а не архив или Parquet.')
        if meta['ContentLength'] > 50*1024**2:
            raise ValueError('Используйте подготовленный фрагмент для большого файла.')
        response = s3.get_object(Bucket=source_bucket, Key=source_key)
        try: display(Audio(response['Body'].read()))
        finally: response['Body'].close()

if not selected.empty:
    raw_row = selected.iloc[example_index]
    raw_passes = current[(current.batch_id == raw_row.batch_id) & (current.sample_id == raw_row.sample_id)]
    for prefix in raw_passes.raw_result_prefix.dropna().unique():
        objects = s3.list_objects_v2(Bucket='speech-annotations', Prefix=prefix+'/', MaxKeys=30)
        display(pd.DataFrame([{'key': x['Key'], 'bytes': x['Size']} for x in objects.get('Contents', [])]))
# inspect_source('speech-corpora', 'staging/...', play=False)
# print(read_text_object(s3, 'speech-annotations', 'ключ из таблицы выше'))


## Сохранённые эксперименты с промптами (по желанию)
Показывает оба опыта на одних примерах, аудио каждого примера — один раз. Новых запросов к модели нет.


In [ ]:
show_prompt_experiments = False  # True — загрузить исторические опыты.
show_exact_prompts = False

def display_prompt_experiments():
    prefixes = ['debug/mixed-language-90bd17c1f3fd', 'debug/precision-triad-6f3319f29d81']
    reports = [(prefix, json.loads(read_text_object(s3, 'speech-annotations', prefix+'/report.json'))) for prefix in prefixes]
    seen = set()
    for _, report in reports:
        for example in report['examples']:
            if example['sample_id'] in seen: continue
            seen.add(example['sample_id'])
            print('Пример:', example['sample_id'])
            print('Замечание пользователя:', example['feedback']['note'])
            print('Подтверждённый текст:', example['feedback']['verbatim_text'])
            print('Исходная метка:', example['original_annotation'])
            for prefix, experiment in reports:
                rows = [r for r in experiment['results'] if r['sample_id'] == example['sample_id']]
                print(prefix)
                display(pd.DataFrame(rows)[['variant','language','transcript','error']])
                if show_exact_prompts:
                    for result in rows:
                        request = json.loads(read_text_object(s3, 'speech-annotations', result['artifact_prefix']+'/request.json'))
                        print(result['variant'], request['prompt'])
            response = s3.get_object(Bucket=example['audio_bucket'], Key=example['audio_key'])
            try:
                if response['ContentLength'] > 100*1024**2: raise ValueError('Слишком большое аудио')
                data = response['Body'].read()
            finally: response['Body'].close()
            assert hashlib.sha256(data).hexdigest() == example['audio_sha256']
            display(Audio(data))
if show_prompt_experiments:
    display_prompt_experiments()
else:
    print('Исторические опыты скрыты. Для просмотра поставьте show_prompt_experiments = True и выполните эту ячейку.')
